<a href="https://colab.research.google.com/github/jromjenine76-alt/pdf-to-video-converter/blob/main/area51-the-silent-runway-20261009/Area51_Free_Cinematic_Movie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Area 51: The Silent Runway — Colab movie maker
Original fictional story inspired by your two Area 51 PDFs. Generates moving footage with **Wan 2.1 1.3B**, narrates with **Kokoro**, and assembles an MP4. No OpenArt, Flow, or paid OpenAI API calls.

**Not rendered or GPU-tested yet.** The code and assembly are checked separately. Free Colab GPU availability, memory, and session time are not guaranteed. Expect substantial model downloads and long render times; a complete film may require several sessions. Character identity can drift between independent shots; this is an experimental movie generator, not a promise of Veo quality.

On your phone: save a copy, select **Runtime → Change runtime type → T4 GPU** (or another GPU), then run the numbered cells in order. If Colab offers only CPU, stop rather than buying compute.

The default renders one opening shot so you can inspect it. To render more, change **SHOTS_THIS_RUN** in cell 2; previously finished shots are reused. Each generated 81-frame clip is slowed to fit its narration, rather than replaced by still pictures.

Source screenplay: [GitHub](https://github.com/jromjenine76-alt/pdf-to-video-converter/blob/movie/area51-the-silent-runway-20261009/productions/area51/The_Silent_Runway_Screenplay.md).
Model documentation: [Wan](https://huggingface.co/docs/diffusers/v0.35.1/api/pipelines/wan), [Kokoro](https://huggingface.co/hexgrad/Kokoro-82M).



## 1. Install the free tools
No API keys or payment details are required. This notebook uses your Google Drive only to preserve completed scenes across disconnected Colab sessions.


In [ ]:
import subprocess, sys
subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "-y", "install", "ffmpeg", "espeak-ng"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "diffusers==0.35.1", "transformers==4.56.2", "accelerate==1.10.1",
    "bitsandbytes==0.47.0", "ftfy", "sentencepiece",
    "imageio[ffmpeg]", "kokoro==0.9.4", "soundfile"], check=True)
print("Tools installed. If Colab requests a restart, restart and continue with cell 2.")




## 2. Save location and shot selection
Mount Drive when prompted. Set SHOTS_THIS_RUN to 2 or 3 after inspecting the opening, and rerun cells 3–6 to progress in small batches. It counts new shots, not ones already finished. It never submits paid jobs.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import json, hashlib, subprocess, gc
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU is assigned. Choose Runtime > Change runtime type > GPU, or try later. Do not continue on CPU.")
print("GPU:", torch.cuda.get_device_name(0))
ROOT = Path("/content/drive/MyDrive/Area51_Silent_Runway")
ROOT.mkdir(parents=True, exist_ok=True)
SHOTS_THIS_RUN = 1  # change to 2 or 3 to progress in batches
VOICE = "am_michael"  # a preset male voice, not a real person's clone
HEIGHT, WIDTH, FRAMES, FPS, STEPS = 480, 832, 81, 16, 30
SHOTS = json.loads("[{\"id\":1,\"visual\":\"Low tracking camera follows a dusty 1950s-style government sedan down an isolated Nevada desert road at dusk. Wheels turn, dust trails, scrub bends in the wind. Mountains surround a distant dry lake.\",\"narration\":\"They sent Mara into the desert to test an aircraft. Nobody mentioned the signal.\"},{\"id\":2,\"visual\":\"Inside the moving car, Mara turns a sealed envelope over in her hands. She watches a distant fence approach through the windshield.\",\"narration\":\"The envelope contained a destination, a clearance number, and one instruction: ask nothing.\"},{\"id\":3,\"visual\":\"At a fictional checkpoint a guard examines the envelope and raises the barrier. Mara drives onward.\",\"narration\":\"No cameras beyond this point. I didn't bring one.\"},{\"id\":4,\"visual\":\"Wide moving shot reveals runway lights across an immense dry lake; sedan approaches low hangars. Title added in editing: AREA 51: THE SILENT RUNWAY. Smaller line: An original fictional story inspired by the supplied PDFs.\",\"narration\":\"Behind the mountains, a hidden world was already awake.\"},{\"id\":5,\"visual\":\"In a hangar, engineers uncover a long, slender, fictional reconnaissance aircraft. Mara walks beneath its wing, looking up.\",\"narration\":\"There were no little green men waiting for her. There were engineers, exhausted pilots, and machines built to outrun danger.\"},{\"id\":6,\"visual\":\"Elias hands Mara a battered stopwatch beside the aircraft.\",\"narration\":\"When the instruments disagree, trust what you can measure. And when the measurements disagree?\"},{\"id\":7,\"visual\":\"Cockpit close-up: Elias fastens his harness; instruments glow; ground crew pulls chocks.\",\"narration\":\"Then measure again.\"},{\"id\":8,\"visual\":\"The reconnaissance aircraft accelerates down the dry-lake runway and lifts into amber sky; dust and heat shimmer move naturally.\",\"narration\":\"For a moment, everything behaved exactly as it should.\"},{\"id\":9,\"visual\":\"Mara watches an analog monitoring console. A second needle begins oscillating while a paper recorder feeds beneath it.\",\"narration\":\"Then a second signal appeared. Not interference. Not noise. A pattern.\"},{\"id\":10,\"visual\":\"Close-up of Mara's hands comparing recorder traces with her stopwatch; she glances sharply toward the radio.\",\"narration\":\"Elias, repeat your last heading. I haven't transmitted.\"},{\"id\":11,\"visual\":\"An operator reaches to switch off the second receiver; Mara gently stops his hand.\",\"narration\":\"We were told to ignore that channel. By someone who knew it was there?\"},{\"id\":12,\"visual\":\"Instrument-light reflection in Mara's eyes, intercut with the aircraft banking above dark mountains.\",\"narration\":\"The signal followed the aircraft. Every turn. Every change in altitude. Always a few seconds ahead.\"},{\"id\":13,\"visual\":\"Elias steadies the controls; his expression changes as he hears Mara through his headset.\",\"narration\":\"You see it too? How long has this been happening?\"},{\"id\":14,\"visual\":\"Mara walks rapidly down a fluorescent corridor with the paper trace in her hand. A locked file-room door stops her.\",\"narration\":\"The locked doors did not frighten her. The silence around them did.\"},{\"id\":15,\"visual\":\"Mara returns to the console and adjusts a receiver dial while the stopwatch ticks.\",\"narration\":\"Cut your transmitter for eight seconds. Let's see what follows.\"},{\"id\":16,\"visual\":\"Cockpit close-up as Elias hesitates, then switches off the transmitter. Ground receiver needle continues moving.\",\"narration\":\"The aircraft went silent. The other signal kept speaking.\"},{\"id\":17,\"visual\":\"A desk lamp illuminates a hand-drawn terrain diagram as Mara marks two intersecting bearings with a pencil.\",\"narration\":\"It's not coming from the aircraft. It's below the ridge.\"},{\"id\":18,\"visual\":\"Mara and Elias, now back on the ground, drive an old utility vehicle along a rough service road under night sky.\",\"narration\":\"They followed it past the runway, past the last light, to a building missing from their briefing.\"},{\"id\":19,\"visual\":\"In a small concrete equipment room, Elias opens a cabinet containing a humming unfamiliar electronic unit, copper coils and vacuum tubes. No aliens or bodies.\",\"narration\":\"They said this room was empty. Then somebody has been lying.\"},{\"id\":20,\"visual\":\"Mara finds recorder strips already showing the exact sequence she measured, with her stopwatch resting beside them.\",\"narration\":\"Inside, she found the pattern again. Printed before the aircraft had taken off.\"},{\"id\":21,\"visual\":\"The machine stops humming. Elias looks toward a closed inner door. Mara raises a hand for silence.\",\"narration\":\"Tell me that's a fault. I can't.\"},{\"id\":22,\"visual\":\"A low pulse resumes; a narrow blue light appears beneath the inner door. Dust trembles on the cabinet.\",\"narration\":\"Perhaps it was a secret experiment. Perhaps the instrument was wrong. Neither explanation opened the door.\"},{\"id\":23,\"visual\":\"Mara slowly reaches for the handle; Elias remains beside her. Close-up: the stopwatch starts moving backward.\",\"narration\":\"This time, we write down everything.\"},{\"id\":24,\"visual\":\"Cut before the door opens to vast desert at dawn; aircraft contrail crosses the pale sky. Fade to title and Fictional story / AI-generated cinematic imagery.\",\"narration\":\"Some mysteries begin with what we don't know. Others begin when we decide to look.\"}]")
MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
STYLE = ("Live-action cinematic fictional Cold War mystery. Restrained 35mm texture, "
    "realistic physical movement, amber desert light and muted blue shadows. "
    "Mara is a woman in her early thirties with short dark bob, olive jacket, cream shirt, "
    "charcoal trousers, brown satchel. Elias is a gray-haired older man in a pale flight suit. "
    "One coherent moving shot; no image slideshow or poster. ")
NEGATIVE = "still image, slideshow, static picture, cartoon, text, captions, watermark, low quality, distorted faces, malformed hands, extra limbs"
def probe(path):
    result = subprocess.run(["ffprobe", "-v", "error", "-show_streams",
        "-show_format", "-of", "json", str(path)], check=True, capture_output=True, text=True)
    return json.loads(result.stdout)
def duration(path):
    return float(probe(path)["format"]["duration"])
def signature(shot):
    return hashlib.sha256(json.dumps({"shot":shot,"model":MODEL_ID,"size":[HEIGHT,WIDTH],
        "frames":FRAMES,"fps":FPS,"steps":STEPS,"style":STYLE,"negative":NEGATIVE,"voice":VOICE},
        sort_keys=True).encode()).hexdigest()
def is_finished(shot):
    p = ROOT / ("shot_%02d.mp4" % shot["id"])
    meta = p.with_suffix(".json")
    try:
        info = json.loads(meta.read_text())
        return (info["signature"] == signature(shot) and duration(p) > 0 and
            any(s["codec_type"] == "video" for s in probe(p)["streams"]))
    except (OSError, ValueError, KeyError, subprocess.CalledProcessError):
        return False
if not isinstance(SHOTS_THIS_RUN, int) or SHOTS_THIS_RUN < 1:
    raise ValueError("SHOTS_THIS_RUN must be a positive integer.")
PENDING = [s for s in SHOTS if not is_finished(s)][:SHOTS_THIS_RUN]
(ROOT / "shot_plan.json").write_text(json.dumps(SHOTS, indent=2))
print("New shots in this run:", [s["id"] for s in PENDING])
print("Finished:", sum(is_finished(s) for s in SHOTS), "/", len(SHOTS))




## 3. Generate the narration locally
One consistent male narrator reads the narration and quoted dialogue. It is not character lip-sync. This step runs Kokoro on CPU so it leaves GPU memory for video.


In [ ]:
import numpy as np
import soundfile as sf
from kokoro import KPipeline
tts = KPipeline(lang_code="a", device="cpu")
for shot in PENDING:
    target = ROOT / ("narration_%02d.wav" % shot["id"])
    pieces = []
    for _, _, audio in tts(shot["narration"], voice=VOICE, speed=0.95):
        pieces.append(np.asarray(audio, dtype=np.float32))
    waveform = np.concatenate(pieces) if pieces else np.zeros(24000, dtype=np.float32)
    sf.write(str(target), waveform, 24000)
    print("Narration ready:", shot["id"], round(len(waveform)/24000, 1), "seconds")
del tts
gc.collect()




## 4. Encode the scene prompts
The large text encoder loads in 4-bit precision to reduce memory use. It is released before the video model loads. If model loading fails or Colab runs out of memory, stop and share the error—there is no paid fallback.


In [ ]:
EMBEDS = {}
if PENDING:
    from transformers import UMT5EncoderModel, BitsAndBytesConfig
    from diffusers import WanPipeline
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float32)
    encoder = UMT5EncoderModel.from_pretrained(MODEL_ID, subfolder="text_encoder",
        quantization_config=quant, torch_dtype=torch.float32, device_map={"": 0})
    encoder_pipe = WanPipeline.from_pretrained(MODEL_ID, text_encoder=encoder,
        transformer=None, vae=None, torch_dtype=torch.float32)
    for shot in PENDING:
        with torch.inference_mode():
            positive, negative = encoder_pipe.encode_prompt(
                prompt=STYLE + shot["visual"], negative_prompt=NEGATIVE,
                do_classifier_free_guidance=True, num_videos_per_prompt=1,
                device=torch.device("cuda"), max_sequence_length=256)
        EMBEDS[shot["id"]] = (positive.cpu(), negative.cpu())
    del encoder_pipe, encoder, positive, negative
    gc.collect()
    torch.cuda.empty_cache()
print("Prompt encoding finished.")




## 5. Generate actual moving footage and mix narration
This is the slow step. Each finished narrated MP4 is saved immediately to Drive. Rerunning later skips completed matching shots. No still-image substitute is used if generation fails.


In [ ]:
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video
if PENDING:
    # Float32 avoids BF16 incompatibility / FP16 overflow on older T4 hardware.
    # Model CPU offload and VAE tiling reduce peak GPU memory use.
    vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
    pipe = WanPipeline.from_pretrained(MODEL_ID, vae=vae, text_encoder=None,
        torch_dtype=torch.float32)
    pipe.enable_model_cpu_offload()
    pipe.vae.enable_tiling()
    for shot in PENDING:
        number = shot["id"]
        print("Generating shot", number, "of", len(SHOTS), flush=True)
        positive, negative = EMBEDS[number]
        with torch.inference_mode():
            frames = pipe(prompt_embeds=positive.to('cuda'), negative_prompt_embeds=negative.to('cuda'),
                height=HEIGHT, width=WIDTH, num_frames=FRAMES, num_inference_steps=STEPS,
                guidance_scale=6.0, generator=torch.Generator("cpu").manual_seed(7000 + number)
            ).frames[0]
        if not np.isfinite(frames).all() or float(np.std(frames)) < 0.001:
            raise RuntimeError("Generation produced invalid or blank frames. No substitute video was created.")
        raw = Path("/content") / ("raw_%02d.mp4" % number)
        export_to_video(frames, str(raw), fps=FPS)
        wav = ROOT / ("narration_%02d.wav" % number)
        length = max(8.0, duration(wav) + 0.6)
        ratio = length / duration(raw)
        target = ROOT / ("shot_%02d.mp4" % number)
        temp = target.with_name(target.stem + ".partial.mp4")
        subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", str(raw), "-i", str(wav),
            "-vf", "setpts=%.8f*PTS,fps=24,scale=832:480,setsar=1" % ratio,
            "-af", "apad", "-t", "%.3f" % length, "-map", "0:v:0", "-map", "1:a:0",
            "-c:v", "libx264", "-preset", "fast", "-crf", "20", "-pix_fmt", "yuv420p",
            "-c:a", "aac", "-ar", "48000", "-ac", "2", "-movflags", "+faststart",
            str(temp)], check=True)
        info = probe(temp)
        if not any(s["codec_type"]=="audio" for s in info["streams"]):
            raise RuntimeError("Narration track is missing; the shot was not marked finished.")
        temp.replace(target)
        target.with_suffix(".json").write_text(json.dumps(
            {"signature":signature(shot), "duration":duration(target)}, indent=2))
        raw.unlink(missing_ok=True)
        del frames
        gc.collect()
        torch.cuda.empty_cache()
        print("Saved shot", number, flush=True)
    del pipe, vae
    EMBEDS.clear()
    gc.collect()
    torch.cuda.empty_cache()




## 6. Preview or download
Creates a preview from the completed consecutive shots. It calls the result a full movie only after all 24 shots exist. Actual length depends on narration duration; there is no speed-reading or automatic cut-off. Downloading can take time on a phone. Narration is AI-generated; characters and plot are fictional.


In [ ]:
from IPython.display import Video, display
from google.colab import files
complete = []
for shot in SHOTS:
    if not is_finished(shot):
        break
    complete.append(ROOT / ("shot_%02d.mp4" % shot["id"]))
if not complete:
    raise RuntimeError("No completed opening shot yet. Finish cell 5 first.")
playlist = ROOT / "concat.txt"
# Only internally generated filenames appear in the concat list.
playlist.write_text("".join("file '%s'\n" % p.name for p in complete))
name = "Area51_Silent_Runway_Movie.mp4" if len(complete)==len(SHOTS) else "Area51_Silent_Runway_Preview.mp4"
movie = ROOT / name
subprocess.run(["ffmpeg", "-y", "-v", "error", "-f", "concat", "-safe", "0",
    "-i", str(playlist), "-c", "copy", "-movflags", "+faststart", str(movie)], check=True)
streams = probe(movie)["streams"]
if not all(any(s["codec_type"]==kind for s in streams) for kind in ["video", "audio"]):
    raise RuntimeError("The assembled movie is missing video or narration.")
print(("FULL MOVIE" if len(complete)==len(SHOTS) else "PARTIAL PREVIEW"),
    "—", len(complete), "/", len(SHOTS), "shots;",
    round(duration(movie), 1), "seconds. Saved in Google Drive.")
def timestamp(seconds):
    total = round(seconds * 1000)
    hours, rest = divmod(total, 3600000)
    minutes, rest = divmod(rest, 60000)
    secs, ms = divmod(rest, 1000)
    return "%02d:%02d:%02d,%03d" % (hours, minutes, secs, ms)
offset, captions = 0.0, []
for i, path in enumerate(complete):
    length = duration(path)
    captions.append("%d\n%s --> %s\n%s\n" %
        (i+1, timestamp(offset), timestamp(offset+length), SHOTS[i]["narration"]))
    offset += length
srt = movie.with_suffix(".srt")
srt.write_text("\n".join(captions))
import zipfile
package = movie.with_suffix(".zip")
with zipfile.ZipFile(package, "w", compression=zipfile.ZIP_STORED) as archive:
    for path in [movie, srt, ROOT / "shot_plan.json"]:
        archive.write(path, arcname=path.name)
    archive.writestr("Read_Me.txt", "Original fictional story. AI-generated footage and narration. "
        "Captions are timed by scene, not word-aligned. "
        + ("Complete movie." if len(complete)==len(SHOTS) else "Partial preview only."))
print("Download package:", package)
display(Video(str(movie), embed=True))
# Run the next line if you want to download now:
# files.download(str(package))


